# CLARA Archiver Image Database
CLARA's diagnostic cameras store many images, and this database allows users to search for images that match target criterea for dataset building.

## Connect

Errors are worth unwrapping properly: the service answers an unknown PV,
unknown camera or malformed filter with `400` and a `detail` message naming the
problem, which is far more useful than the status code alone.

In [ ]:
import os

import httpx #requests also works identically

BASE_URL = os.environ.get("ARCHIVER_API_URL", "http://apml1.dl.ac.uk:9876")
client = httpx.Client(base_url=BASE_URL, timeout=30.0)


class ApiError(RuntimeError):
    pass


def call(method, path, **kwargs):
    """Issue a request and surface the service's own error message."""
    response = client.request(method, path, **kwargs)
    if response.is_error:
        try:
            detail = response.json().get("detail", response.text)
        except ValueError:
            detail = response.text
        raise ApiError(f"{response.status_code} {method} {path}: {detail}")
    return response.json()


get = lambda path, **kw: call("GET", path, **kw)
post = lambda path, **kw: call("POST", path, **kw)

print(f"talking to {BASE_URL}")

## What is in there

`GET /api/meta` is the orientation call: row count, time bounds, PV count and
the camera list. It doubles as a liveness check.

Interactive documentation for every endpoint is served at `/docs`.

In [ ]:
meta = get("/api/meta")

print(f"{meta['n_rows']} rows, {meta['pv_count']} PVs")
print(f"schema v{meta['schema_version']}, built {meta['created_utc']}")
print(f"{meta['time_from']}  ..  {meta['time_to']}")
print()
for cam in get("/api/cameras"):
    print(f"{cam['name']:<22} {cam['area']:<4} {cam['n_rows']:>6} rows   {cam['first']} .. {cam['last']}")

## Finding a PV

`GET /api/pvs` filters the catalogue by `q` (substring), `area`, `family` and
`field`. Each entry carries the min and max recorded at ingest, which is how to
choose a threshold that will actually match something.

In [ ]:
quads = get("/api/pvs", params={"family": "MAG", "field": "SETI", "q": "QUAD"})
for pv in quads[:8]:
    print(f"{pv['name']:<28} {pv['area']:<4} {pv['min']:>10.3f} .. {pv['max']:>8.3f}")
print(f"... {len(quads)} matching PV(s)")

# Pick one that actually varies, so the queries below are not trivially true.
MAGNET = next((pv for pv in quads if pv["min"] != pv["max"]), quads[0])
LO, HI = MAGNET["min"], (MAGNET["min"] + MAGNET["max"]) / 2
print(f"\nusing {MAGNET['name']}, querying {LO} .. {HI}")

## Query 1 — the flat form

`GET /api/search` takes the whole query as query-string parameters, so it is
shareable as a URL and reproducible with `curl`.

A filter is `f=<pv>|<op>|<v1>[|<v2>]`. The separator is `|` rather than `:`
because PV names contain `:` and `-`. Repeat `f` for more filters and join them
with `combine=and|or`.

Operators: `eq ne lt lte gt gte between outside in isnan notnan`.

In [ ]:
def show(result, *, n=5):
    """Print the headline counters and the first few rows of a search result."""
    print(
        f"{result['total']} row(s) · scanned {result['scanned_rows']} · "
        f"{result['pv_reads']} PV read(s) · {result['elapsed_ms']} ms"
    )
    for row in result["rows"][:n]:
        values = "  ".join(
            f"{name.split(':')[-1]}={value:.3f}" if value is not None else f"{name}=nan"
            for name, value in row["values"].items()
        )
        print(f"  {row['id']:>6}  {row['timestamp']}  {row['camera']:<22} {values}")


params = {
    "f": f"{MAGNET['name']}|between|{LO}|{HI}",
    "select": MAGNET["name"],
    "limit": 5,
}
q1 = get("/api/search", params=params)
show(q1)

print(f"\n{client.build_request('GET', '/api/search', params=params).url}")

Every response reports `scanned_rows` — how far the time window narrowed things
before any PV was read — and `pv_reads`, the number of column slices the server
fetched from HSDS. They make it obvious when a query is touching more than it
should. Note that `elapsed_ms` is the server's own time and excludes the
network hop to you.

## Query 2 — time and cameras

All times are UTC and a bare date is accepted. `from`/`to` given the wrong way
round are swapped rather than rejected, and `days` is the "N days following X"
form so you do no date arithmetic.

`limit=0` returns the count with no rows, which is the cheap way to size a
query before asking for it. `camera` and `area` are repeatable.

In [ ]:
first_day = meta["time_from"][:10]

windowed = get("/api/search", params={"from": first_day, "days": 1, "limit": 0})
print(f"the day from {first_day}:")
show(windowed)

area = get("/api/cameras")[0]["area"]
by_area = get("/api/search", params={"area": area, "limit": 0})
print(f"\narea {area} (no window, and no PV touched):")
show(by_area)

## Query 3 — the structured form

The flat form cannot express nesting. `POST /api/search` takes the query as
JSON, where `filters` is a tree of clauses and groups that nests arbitrarily.

A `pv_pattern` is a glob over several PVs: `"match": "any"` matches the row if
any of them satisfies the predicate, `"all"` if all do. `eq` compares with an
absolute tolerance (`tol`, default `1e-9`) because these are floats read back
from a control system.

In [ ]:
query = {
    "filters": {
        "op": "and",
        "clauses": [
            {"pv": MAGNET["name"], "op": "between", "value": [LO, HI]},
            {
                "op": "or",
                "clauses": [
                    {"pv_pattern": "*:getPhase", "match": "any", "op": "notnan"},
                    {"pv_pattern": "*:getPower", "match": "all", "op": "isnan"},
                ],
            },
        ],
    },
    "select": [MAGNET["name"]],
    "sort": {"by": "timestamp", "order": "desc"},
    "limit": 5,
}

show(post("/api/search", json=query))

## Query 4 — a time series

`/api/series` returns one trace per PV over the matching rows, accepting the
same time, camera and filter clauses. Traces longer than `max_points` are
min/max decimated server-side and flagged with `decimated`, so asking for a
wide window does not mean downloading it all.

`t` holds ISO-8601 timestamps and `v` the values, with `null` where the reading
was NaN — ready to hand to a plotting library.

In [ ]:
result = post(
    "/api/series",
    json={
        "pv_pattern": "*:getPhase",
        "time": {"from": first_day, "days": 1},
        "max_points": 500,
    },
)

print(f"{result['total']} matching row(s) · {result['elapsed_ms']} ms")
for trace in result["series"]:
    values = [v for v in trace["v"] if v is not None]
    span = f"{min(values):>9.2f} .. {max(values):>8.2f}" if values else "(all nan)"
    print(f"  {trace['pv']:<30} {len(trace['t']):>5} pts  {span}  decimated={trace['decimated']}")

## One row in full

`GET /api/rows/{id}` returns a single capture with every PV attached, plus the
path of the capture file holding its images.

In [ ]:
row = get(f"/api/rows/{q1['rows'][0]['id']}")

print(f"{row['camera']}  {row['timestamp']}  ({row['n_images']} image(s))")
print(f"source  {row['source_file']}")
print(f"capture {row['capture_path']}")
print(f"\n{len(row['values'])} PV values, e.g.")
for name in sorted(row["values"])[:5]:
    print(f"  {name:<30} {row['values'][name]}")

**Images are referenced, never served.** There is no endpoint that returns
pixels. The capture files live on the CLARA storage and are not reachable from
the service, so a row gives you the path and the image count; the datasets
inside are named `Capture000001` … `Capture{n_images:06d}`:

```python
import h5py

with h5py.File(row["capture_path"]) as f:
    image = f["Capture000001"][:]   # greyscale
```

That path only opens from a machine that mounts the CLARA storage, which is
probably not the one running this notebook.

## Bulk retrieval

`GET /api/export.csv` streams every matching row, ignoring the page limit. It
takes the same flat parameters as `GET /api/search`, and `select` adds a column
per PV. Stream it rather than buffering — against a real archive this is the
one endpoint that can return a great deal of data.

In [ ]:
import codecs
import csv

params = {"from": first_day, "days": 1, "select": MAGNET["name"]}

with client.stream("GET", "/api/export.csv", params=params) as response:
    response.raise_for_status()
    decoder = codecs.getincrementaldecoder("utf-8")()
    lines = (decoder.decode(chunk) for chunk in response.iter_bytes())
    reader = csv.reader(line for text in lines for line in text.splitlines())
    header = next(reader)
    rows = list(reader)

print(f"columns: {header}")
print(f"{len(rows)} data row(s)")
for row in rows[:3]:
    print(f"  {row}")

## When a query is wrong

An unknown PV, an unknown camera or a malformed filter is a `400` naming the
problem — an unknown camera is an error, not an empty result, on the grounds
that silently returning nothing is the worse failure.

In [ ]:
for bad in (
    {"f": "CLA-NO-SUCH-PV-01:SETI|gt|1"},
    {"camera": "CLA-S99-DIA-CAM-99"},
    {"f": f"{MAGNET['name']}|between|1"},
    {"f": f"{MAGNET['name']}|gt|banana"},
):
    try:
        get("/api/search", params={**bad, "limit": 0})
    except ApiError as exc:
        print(exc)

## Paging

A page is capped at the server's `max_limit` (default 1000); asking for more is
a `400` pointing you at the CSV export. `total` is the full match count
regardless of the page, so it is all you need to walk the results.

In [ ]:
PAGE = 50
offset, seen = 0, 0

while True:
    page = get("/api/search", params={"area": area, "limit": PAGE, "offset": offset})
    seen += len(page["rows"])
    print(f"offset {offset:>5}: {len(page['rows']):>3} row(s) of {page['total']}")
    offset += PAGE
    if offset >= page["total"]:
        break

print(f"{seen} row(s) total")

In [ ]:
client.close()

## FastAPI

There's a FastAPI doc at `ARCHIVER_API_URL/docs` which exposes all endpoints. Feel free to look for more features.
Note that while it is possible to get images from the API, this is for viewing only, not for analysis. Analysis should be performed on the raw HDF5 files which can be found on the data server.